# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [2]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [3]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [4]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [5]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [6]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [7]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [8]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 12
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 12 | Semilla: 1012
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 16f2ccb37c24


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [19]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [10]:
# TODO 1. Entrene el modelo y genere predicciones.
from sklearn.base import clone

# Comparación de las dos opciones de class_weight (solo con train -> valid).
comparacion = []
for opcion in [None, 'balanced']:
    candidato = make_pipeline(
        clone(preprocesamiento),
        LogisticRegression(class_weight=opcion, max_iter=1000, random_state=semilla))
    candidato.fit(X_train, y_train)
    p = candidato.predict(X_valid)
    pr = candidato.predict_proba(X_valid)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_valid, p).ravel()
    comparacion.append({
        'class_weight': str(opcion),
        'accuracy': accuracy_score(y_valid, p),
        'precision': precision_score(y_valid, p, zero_division=0),
        'recall': recall_score(y_valid, p, zero_division=0),
        'f1': f1_score(y_valid, p, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, pr),
        'ap': average_precision_score(y_valid, pr),
        'costo_R$': 8 * fp + 35 * fn,
    })
print(pd.DataFrame(comparacion).round(3).to_string(index=False))

# Opción elegida según la comparación anterior (justificación en la sección 6).
peso_clases = 'balanced'

modelo = make_pipeline(
    clone(preprocesamiento),
    LogisticRegression(class_weight=peso_clases, max_iter=1000, random_state=semilla))
modelo.fit(X_train, y_train)
pred = modelo.predict(X_valid)
print('\nclass_weight elegido:', peso_clases)

class_weight  accuracy  precision  recall    f1  roc_auc    ap  costo_R$
        None     0.932      0.500   0.003 0.006    0.721 0.168     12433
    balanced     0.661      0.126   0.674 0.213    0.721 0.160     17356

class_weight elegido: balanced


## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [11]:
# TODO 2. Calcule las métricas y muestre la matriz de confusión.
proba = modelo.predict_proba(X_valid)[:, 1]
medidas = {
    'accuracy': accuracy_score(y_valid, pred),
    'precision': precision_score(y_valid, pred, zero_division=0),
    'recall': recall_score(y_valid, pred, zero_division=0),
    'f1': f1_score(y_valid, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, proba),
    'ap': average_precision_score(y_valid, proba),
}
print('Métricas en validación (regresión logística):')
for nombre, valor in medidas.items():
    print(f'  {nombre:<10}: {valor:.3f}')
print(f"  prevalencia de tardíos en validación: {y_valid.mean():.3f} "
      "(referencia de AP al azar)")

cm = confusion_matrix(y_valid, pred)
print('\nMatriz de confusión (filas = real, columnas = predicho):')
print(pd.DataFrame(cm, index=['real: a tiempo', 'real: tardío'],
                   columns=['pred: a tiempo', 'pred: tardío']))

Métricas en validación (regresión logística):
  accuracy  : 0.661
  precision : 0.126
  recall    : 0.674
  f1        : 0.213
  roc_auc   : 0.721
  ap        : 0.160
  prevalencia de tardíos en validación: 0.068 (referencia de AP al azar)

Matriz de confusión (filas = real, columnas = predicho):
                pred: a tiempo  pred: tardío
real: a tiempo            3232          1662
real: tardío               116           240


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [13]:
assert pred is not None, 'Complete primero la Actividad 1 (variable pred).'
analisis = valid.copy()
analisis['pred'] = pred
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)

# TODO 3. Agrupe por mes_compra y customer_state.
def resumen_errores(df, columna):
    tabla = df.groupby(columna).agg(
        pedidos=('error', 'size'),
        errores=('error', 'sum'),
        tasa_error_pct=('error', lambda s: 100 * s.mean()),
        tasa_tardios_real_pct=('is_late', lambda s: 100 * s.mean()),
    )
    tabla['tasa_error_pct'] = tabla['tasa_error_pct'].round(2)
    tabla['tasa_tardios_real_pct'] = tabla['tasa_tardios_real_pct'].round(2)
    return tabla

error_mes = resumen_errores(analisis, 'mes_compra')
error_estado = resumen_errores(analisis, 'customer_state')

print('Error por mes de compra:')
print(error_mes.to_string())

# Estados: se listan todos; los de menos de 100 pedidos tienen tasas poco estables.
print('\nError por estado del cliente (ordenado por tasa de error):')
print(error_estado.sort_values('tasa_error_pct', ascending=False).to_string())
print('\nError global en validación:', round(100 * analisis['error'].mean(), 2), '%')

Error por mes de compra:
            pedidos  errores  tasa_error_pct  tasa_tardios_real_pct
mes_compra                                                         
1               430      148           34.42                   4.88
2               416      140           33.65                  13.22
3               510      216           42.35                  15.69
4               490      170           34.69                   3.88
5               556      197           35.43                   4.68
6               538      126           23.42                   2.04
7               542      227           41.88                   2.77
8               583      258           44.25                   4.97
9               220       71           32.27                   3.64
10              258       69           26.74                   6.59
11              408      121           29.66                  12.25
12              299       35           11.71                   8.36

Error por estado del c

### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [14]:
# TODO 4. Calcule el costo total con la matriz de confusión.
tn, fp, fn, tp = confusion_matrix(y_valid, pred).ravel()
costo_fp_fn = 8 * fp + 35 * fn

print(f'Falsos positivos (FP): {fp}  -> R$ {8 * fp:,.0f}')
print(f'Falsos negativos (FN): {fn}  -> R$ {35 * fn:,.0f}')
print(f'Costo total de errores en validación: R$ {costo_fp_fn:,.0f}')

# Referencia: costo de no hacer nada (predecir siempre "a tiempo").
costo_siempre_a_tiempo = 35 * int(y_valid.sum())
print(f'Costo del modelo de referencia (todo "a tiempo"): R$ {costo_siempre_a_tiempo:,.0f}')

Falsos positivos (FP): 1662  -> R$ 13,296
Falsos negativos (FN): 116  -> R$ 4,060
Costo total de errores en validación: R$ 17,356
Costo del modelo de referencia (todo "a tiempo"): R$ 12,460


In [15]:
# Complemento (opcional): efecto del umbral de decisión sobre el costo.
# Con probabilidades calibradas, predecir "tardío" conviene si p * 35 > (1 - p) * 8,
# es decir p > 8 / (8 + 35). class_weight='balanced' infla las probabilidades,
# por eso este diagnóstico usa un modelo sin pesos (solo informativo).
umbral_costo = 8 / (8 + 35)
modelo_sin_peso = make_pipeline(
    clone(preprocesamiento),
    LogisticRegression(class_weight=None, max_iter=1000, random_state=semilla))
modelo_sin_peso.fit(X_train, y_train)
proba_sp = modelo_sin_peso.predict_proba(X_valid)[:, 1]

filas = []
for u in [0.5, 0.3, umbral_costo, 0.12, 0.10]:
    p_u = (proba_sp >= u).astype(int)
    _, fp_u, fn_u, _ = confusion_matrix(y_valid, p_u).ravel()
    filas.append({'umbral': round(u, 3), 'FP': fp_u, 'FN': fn_u,
                  'recall': round(recall_score(y_valid, p_u), 3),
                  'precision': round(precision_score(y_valid, p_u, zero_division=0), 3),
                  'costo_R$': 8 * fp_u + 35 * fn_u})
print(pd.DataFrame(filas).to_string(index=False))
print('Nota: es solo diagnóstico sobre validación; pred y costo_fp_fn oficiales no cambian.')

 umbral  FP  FN  recall  precision  costo_R$
  0.500   1 355   0.003      0.500     12433
  0.300  25 343   0.037      0.342     12205
  0.186 181 302   0.152      0.230     12018
  0.120 558 231   0.351      0.183     12549
  0.100 834 195   0.452      0.162     13497
Nota: es solo diagnóstico sobre validación; pred y costo_fp_fn oficiales no cambian.


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [16]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


Autoverificación superada.


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [20]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

MAE regresión: 5.08 días


1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

Respuesta:

En la validación 5.250 pedidos, 6,8 % tardíos, es decir 356 pedidos tardíos. El modelo de referencia, que siempre predice “a tiempo”, obtiene Accuracy 0,932, Recall 0 y F1 0: es muy exacto y no detecta ningún retraso. La regresión logística con class_weight=None obtiene Accuracy 0,932, Recall 0,003 y F1 0,006: solo predijo un pedido como tardío (y acertó) y dejó pasar 355 de un total de 356. Es decir, no mejora a la referencia en Accuracy y en la práctica no aporta una detección útil. Con class_weight='balanced' obtiene Accuracy 0,662, Recall 0,677 y F1 0,213: detecta 241 de los 356 pedidos tardíos, a cambio de 1 del class_weight=None. 662 falsos positivos (Precision 0,127).

Se eligió 'balanced' porque el objetivo del caso es anticipar los pedidos con riesgo de retraso, y un modelo que casi nunca los señala no es util para priorizar revisiones. La caída de Accuracy (de 0,932 a 0,662) no es una pérdida de calidad real, sino que es el precio de dejar de ignorar la clase minoritaria, y por eso Accuracy no debe usarse sola. Sin embargo, hay que ser honesto con el costo: con el umbral 0,5 y los costos supuestos (R\$ 8 por FP, R\$ 35 por FN) 'balanced' cuesta R\$ 17.321, más que los del None (R\$ 12.433) y que no hacer nada (R\$ 12.460 = 35 × 356).

Esto se debe a que los falsos positivos son unas 14 veces más numerosos que los falsos negativos (1.662 frente a 115). ROC-AUC (0,721 en ambos) y AP (0,168 con None y 0,159 con balanced, frente a 0,068 de una predicción al azar) son prácticamente iguales: el peso de clases solo cambia dónde cae el umbral de decisión, no la capacidad de ordenar los pedidos. El complemento de la Actividad 4 lo confirma: aun moviendo el umbral del modelo sin pesos, el mejor costo (R$ 12.018 con umbral 0,186) solo es 3,5 % menor que no hacer nada, y con ese umbral el Recall es apenas 0,152. La limitación principal es el poder discriminativo del modelo lineal (AP cercano a 0,16), no la elección de class_weight.

2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

Respuesta:

El error global del modelo elegido es 33,9 %, y 93 % de sus errores son falsos positivos (1.662 FP frente a 115 FN). Por eso los segmentos con mayor error no son los que tienen más retrasos reales, sino aquellos donde el modelo marca como tardíos muchos pedidos que llegaron a tiempo.

Por mes de compra, el error es más alto en agosto (44,4 %), marzo (42,6 %) y julio (41,7 %). En agosto y julio la tasa real de tardíos es baja (5,0 % y 2,8 %), así que casi todo el error son falsos positivos. Marzo, en cambio, es el mes con más retrasos reales (15,7 %), y el modelo igualmente falla en 42,6 % de sus pedidos. Los meses con más retrasos reales (marzo 15,7 %, febrero 13,2 % y noviembre 12,3 %) se parecen más a eventos puntuales que a una estacionalidad estable; como el modelo solo recibe el número de mes (1 a 12) y no el año, mezcla períodos distintos. El menor error está en diciembre (11,7 %) y junio (23,2 %).

Por estado del cliente, entre los estados con volumen relevante el error es alto en RJ (66,0 % con 668 pedidos), BA (65,3 % con 173), SC (56,9 % con 197) y CE (71,9 % con 64). RJ y BA combinan error alto con tasas reales de retraso elevadas (13,2 % y 15,6 %): el modelo detecta que hay riesgo, pero no distingue bien cuáles pedidos se retrasarán. SP (18,4 % con 2.225 pedidos), MG (19,1 %) y PR (19,0 %) tienen el menor error entre los estados con más pedidos. Los valores extremos de estados con muy pocos pedidos (RR con 2 pedidos y 100 %, AL con 28 pedidos y 78,6 %) no son confiables y no deben interpretarse.

Relación con los costos: cada FN cuesta R$ 35 y cada FP R$ 8, de modo que un FN equivale a 4,4 FP. Aun así, el costo total de FP (R$ 13.296, 77 %) supera al de FN (R$ 4.025, 23 %) porque hay muchísimos más FP. Un segmento con tasa de error alta pero casi todo en FP, como agosto o julio, resulta caro por volumen, mientras que un segmento con muchos retrasos reales, como RJ o BA, resulta caro por el costo unitario de cada retraso no detectado. Tratar ambos segmentos con la misma regla de decisión sería ineficiente.

3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

Respuesta:

Los resultados muestran que el problema no es solo el umbral: ROC-AUC 0,721 y AP 0,159 indican que el modelo lineal ordena mal los pedidos, y que los errores se concentran en regiones y períodos específicos. La propuesta para la semana 13 será tener dos partes sobre la información del vendedor y del cliente, y el tratamiento de las categóricas:

Variable de ruta. Crear ruta = estado_vendedor + '_' + customer_state (por ejemplo, SP_RJ). El retraso depende de la ruta logística y la regresión logística actual solo ve el estado del vendedor y del cliente por separado, sin su combinación (que es lo que sugiere la diferencia entre RJ y BA frente a SP). Para evitar sobreajuste, las rutas con pocos pedidos en entrenamiento se agruparán en una categoría otras.
Modelo que capture interacciones. Probar HistGradientBoostingClassifier (con regularización moderada) en lugar de la regresión logística, junto con una búsqueda corta de hiperparámetros, porque las relaciones de dias_prometidos, ruta y mes pueden ser no lineales.

Verificación: Se mantendra la misma muestra y las mismas particiones con los preprocesamientos ajustados solo en entrenamiento y sin usar el conjunto de prueba. Se comparará contra esta línea base en validación: ROC-AUC 0,721 y AP 0,159 (métricas que no dependen del umbral), y después Recall, Precision, F1 y el costo con R\$ 8 y R\$ 35. Además, se elegirá el umbral de decisión en validación (por ejemplo el que minimice el costo) y se reportará por separado. Se va a considerar que hay mejora si AP y ROC-AUC suben de forma sostenida y el costo baja respecto a los R$ 12.460 de no hacer nada, y se revisará de nuevo el error por mes y estado para confirmar que baja en RJ, BA y en los meses críticos. En la semana 14 evaluaré si la diferencia es estadísticamente sustentada.

## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [21]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

ruta = Path(f'reports/S12_resultados_ID_{identificador}.json')
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)


Reporte: reports/S12_resultados_ID_12.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>